In [52]:
import os
import zipfile
import requests
import numpy as np
import pandas as pd
import plotly.express as px
import folium


from pathlib import Path

In [53]:
CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/JC/"
PERIOD = "202510"

In [54]:
file_name = f"JC-{PERIOD}-citibike-tripdata.zip"
url = f"{CITIBIKE_INDEX_URL}/{file_name}"
url

'https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip'

In [55]:
# eval: true
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile

file_name = f"JC-{PERIOD}-citibike-tripdata.zip"

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)

zip_path = output_dir / file_name


## Dowloading the Zip file
url = f"{CITIBIKE_INDEX_URL}/{file_name}"

print(f"Downloading: {url}")
urlretrieve(url, zip_path)

print(f"Saved ZIP file to: {zip_path}")


with ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(output_dir)

print(f"Extracted files into: {output_dir}")

Downloading: https://s3.amazonaws.com/tripdata/JC-202510-citibike-tripdata.zip
Saved ZIP file to: ../data/JC/JC-202510-citibike-tripdata.zip
Extracted files into: ../data/JC


In [56]:
zip_path.unlink()
print("ZIP file removed.")

ZIP file removed.


In [57]:
file_name  = 'JC-202510-citibike-tripdata.csv'
citibike_202510 = pd.read_csv(f'{output_dir}/{file_name}')
citibike_202510.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,929852F0DC04F49C,electric_bike,2025-10-08 06:59:53.767,2025-10-08 07:04:08.012,Warren St,JC006,Essex Light Rail,NaN,40.721124,-74.038051,NaN,NaN,member
1,7E6F8FADF45B1D8E,electric_bike,2025-10-08 08:13:15.096,2025-10-08 08:18:16.168,Newark Ave,JC032,Essex Light Rail,NaN,40.721525,-74.046305,NaN,NaN,member
2,9B679869FCA8F845,electric_bike,2025-10-09 10:10:26.020,2025-10-09 10:20:08.505,Liberty Light Rail,JC052,Newport PATH,NaN,40.711242,-74.055701,NaN,NaN,member
3,E2051DD457BC4076,electric_bike,2025-10-09 11:34:22.937,2025-10-09 11:48:49.501,Hoboken Ave at Monmouth St,JC105,Newport PATH,NaN,40.735208,-74.046964,NaN,NaN,member
4,239AC07371E414EC,electric_bike,2025-10-23 19:24:16.237,2025-10-23 19:28:22.592,Union St & Bergen Ave,JC122,Garfield Light Rail,JC119,40.715750,-74.078870,40.710526,-74.070112,member


In [58]:
def period_iterator(year:list,start_m:int, stop_m:int)->list:
    """
    
    """
    YEAR = year
    MONTH =  [str(i+1) if i+1>9 else "0" + str(i+1) for i in range(start_m, stop_m)]

    periods = []

    for i in YEAR:
        for j in MONTH:
            k = i+j
            periods.append(k)
    # print(periods)
    return periods

In [59]:
PERIODS = period_iterator(year = ['2025'],start_m = 1, stop_m = 12)

PERIODS

['202502',
 '202503',
 '202504',
 '202505',
 '202506',
 '202507',
 '202508',
 '202509',
 '202510',
 '202511',
 '202512']

In [60]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError

CITIBIKE_INDEX_URL = "https://s3.amazonaws.com/tripdata"
OUTPUT_DIR = "../data/JC"
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(exist_ok=True)


for i in PERIODS:

    try:
        file_name = f"JC-{i}-citibike-tripdata.csv.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    except (HTTPError, URLError, FileNotFoundError):
        file_name = f"JC-{i}-citibike-tripdata.zip"
        url = f"{CITIBIKE_INDEX_URL}/{file_name}"

        zip_path = output_dir / file_name
        urlretrieve(url, zip_path)

    with ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(output_dir)
    print(f'{file_name}  Extracted')
    zip_path.unlink()
    print(f"{file_name} removed.")


JC-202502-citibike-tripdata.csv.zip  Extracted
JC-202502-citibike-tripdata.csv.zip removed.
JC-202503-citibike-tripdata.csv.zip  Extracted
JC-202503-citibike-tripdata.csv.zip removed.
JC-202504-citibike-tripdata.csv.zip  Extracted
JC-202504-citibike-tripdata.csv.zip removed.
JC-202505-citibike-tripdata.csv.zip  Extracted
JC-202505-citibike-tripdata.csv.zip removed.
JC-202506-citibike-tripdata.csv.zip  Extracted
JC-202506-citibike-tripdata.csv.zip removed.
JC-202507-citibike-tripdata.csv.zip  Extracted
JC-202507-citibike-tripdata.csv.zip removed.
JC-202508-citibike-tripdata.csv.zip  Extracted
JC-202508-citibike-tripdata.csv.zip removed.
JC-202509-citibike-tripdata.csv.zip  Extracted
JC-202509-citibike-tripdata.csv.zip removed.
JC-202510-citibike-tripdata.zip  Extracted
JC-202510-citibike-tripdata.zip removed.
JC-202511-citibike-tripdata.csv.zip  Extracted
JC-202511-citibike-tripdata.csv.zip removed.
JC-202512-citibike-tripdata.csv.zip  Extracted
JC-202512-citibike-tripdata.csv.zip remov

In [61]:
import glob
import numpy as np
import pandas as pd

file_names = glob.glob(f'{output_dir}/*.csv')



dfs = []
cols = []
for file_name in file_names:
    df = pd.read_csv(file_name)
    print(df.columns, 2*"||",len(df.columns))

    cols.append(list(df.columns))
    dfs.append(df)

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str') |||| 13
Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      d

In [62]:
citibike_df = pd.concat(dfs)
citibike_df.head()

,ride_id,rideable_type,started_at,ended_at,start_station_name,start_station_id,end_station_name,end_station_id,start_lat,start_lng,end_lat,end_lng,member_casual
0,9F734BE1BFC45FF4,electric_bike,2025-11-18 18:34:14.943,2025-11-18 18:47:33.391,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member
1,B6C773B13AC0E465,classic_bike,2025-11-26 16:29:15.513,2025-11-26 16:43:45.235,Glenwood Ave,JC094,West Side Ave & Stegman Pkwy,JC131,40.727551,-74.071061,40.710870,-74.093680,member
2,C300465AA158280F,electric_bike,2025-11-04 22:31:58.010,2025-11-04 22:38:57.017,Bloomfield St & 15 St,HB203,Marshall St & 2 St,HB408,40.754530,-74.026580,40.740802,-74.042521,member
3,31A424FC97C8AAFB,classic_bike,2025-11-08 06:51:57.424,2025-11-08 06:57:45.627,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member
4,08C5EA04CB1FDC57,classic_bike,2025-11-24 20:31:21.758,2025-11-24 20:38:01.261,Clinton St & 7 St,HB303,Marshall St & 2 St,HB408,40.745420,-74.033320,40.740802,-74.042521,member


In [64]:
citibike_df.to_csv("../data/JC/JC2025.csv", index = False)

In [65]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'],errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'],errors="coerce")

In [66]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

In [67]:
citibike_df.shape

(952093, 13)